In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/gauravchhajed/prunning-dataset/data.yaml
/kaggle/input/datasets/gauravchhajed/prunning-dataset/yolov8n.pt
/kaggle/input/datasets/gauravchhajed/prunning-dataset/yolo26n.pt
/kaggle/input/datasets/gauravchhajed/prunning-dataset/yolo_dataset/labels/val.cache
/kaggle/input/datasets/gauravchhajed/prunning-dataset/yolo_dataset/labels/train.cache
/kaggle/input/datasets/gauravchhajed/prunning-dataset/yolo_dataset/labels/val/161056d.txt
/kaggle/input/datasets/gauravchhajed/prunning-dataset/yolo_dataset/labels/val/170707v.txt
/kaggle/input/datasets/gauravchhajed/prunning-dataset/yolo_dataset/labels/val/100138d.txt
/kaggle/input/datasets/gauravchhajed/prunning-dataset/yolo_dataset/labels/val/161061v.txt
/kaggle/input/datasets/gauravchhajed/prunning-dataset/yolo_dataset/labels/val/170039d.txt
/kaggle/input/datasets/gauravchhajed/prunning-dataset/yolo_dataset/labels/val/150447v.txt
/kaggle/input/datasets/gauravchhajed/prunning-dataset/yolo_dataset/labels/val/170183h.txt
/kaggl

In [ ]:
# =========================================
# YOLOv8 STRUCTURED PRUNING (25%) FINAL FIX
# =========================================

!pip install ultralytics -q

import os
import shutil
import yaml
from ultralytics import YOLO

# =========================================
# PATHS
# =========================================
DATASET_INPUT = "/kaggle/input/datasets/gauravchhajed/prunning-dataset/yolo_dataset"
WORK_DIR = "/kaggle/working/yolo_dataset"
DATA_YAML = WORK_DIR + "/data.yaml"

# =========================================
# STEP 1: COPY DATASET
# =========================================
if not os.path.exists(WORK_DIR):
    shutil.copytree(DATASET_INPUT, WORK_DIR)

# Fix YAML
original_yaml = "/kaggle/input/datasets/gauravchhajed/prunning-dataset/data.yaml"

with open(original_yaml, 'r') as f:
    data = yaml.safe_load(f)

data['path'] = WORK_DIR

with open(DATA_YAML, 'w') as f:
    yaml.dump(data, f)

print("✅ Dataset ready")

# =========================================
# STEP 2: DEFINE BASE YOLOv8n YAML (MANUAL)
# =========================================
base_yaml = {
    'nc': 3,
    'scales': {
        'n': [0.33, 0.25, 1024]
    },
    'backbone': YOLO("yolov8n.pt").model.yaml['backbone'],
    'head': YOLO("yolov8n.pt").model.yaml['head']
}

# =========================================
# STEP 3: CREATE PRUNED YAML (25%)
# =========================================
def create_pruned_yaml():

    new_yaml = base_yaml.copy()

    depth, width, max_channels = new_yaml['scales']['n']

    scale_factor = 0.9  # 25% pruning

    new_yaml['scales']['n'] = [
        depth,
        width * scale_factor,
        int(max_channels * scale_factor)
    ]

    save_path = "/kaggle/working/yolov8n_pruned_25.yaml"

    with open(save_path, 'w') as f:
        yaml.dump(new_yaml, f)

    print("✅ Pruned YAML created")
    return save_path


PRUNED_YAML = create_pruned_yaml()

# =========================================
# STEP 4: PARAMETER CHECK
# =========================================
print("\n🔍 Checking parameters BEFORE training...")

orig_model = YOLO("yolov8n.pt")
pruned_model = YOLO(PRUNED_YAML)

orig_params = sum(p.numel() for p in orig_model.model.parameters())
pruned_params = sum(p.numel() for p in pruned_model.model.parameters())

print(f"\nOriginal Params: {orig_params/1e6:.2f}M")
print(f"Pruned Params:   {pruned_params/1e6:.2f}M")

if pruned_params >= orig_params:
    print("\n❌ ERROR: Pruning not applied")
else:
    print("\n✅ Pruning SUCCESSFUL")

# =========================================
# STEP 5: TRAIN
# =========================================
print("\n🚀 Training...")

model = YOLO(PRUNED_YAML)

model.train(
    data=DATA_YAML,
    epochs=75,
    imgsz=640,
    batch=16,
    device=0,
    lr0=0.001,
    name="pruned_25"
)

# =========================================
# STEP 6: EVALUATE
# =========================================
print("\n📊 Evaluating...")

model = YOLO("runs/detect/pruned_25/weights/best.pt")
model.val(data=DATA_YAML)

print("\n✅ DONE")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Dataset ready
✅ Pruned YAML created

🔍 Checking parameters BEFORE training...

Original Params: 3.16M
Pruned Params:   2.42M

✅ Pruning SUCCESSFUL

🚀 Training...
Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/data.yaml, degrees=0.0, deterministic=Tr